In [ ]:
!pip install polars

In [ ]:
import polars as pl
import seaborn as sns
import matplotlib.pyplot as plt
import math
import pandas as pd

## Read data from S3 and join

In [ ]:
source_bucket = 's3://msds-26.2-data'
file_names = ['sanitized_2022.csv', 'sanitized_2023.csv', 'sanitized_2024.csv', 'sanitized_2025.csv']

In [ ]:
# Read csv from s3 and concat
df_recovery = pl.DataFrame()
for name in file_names:
    file_path = f'{source_bucket}/{name}'
    df_name = pl.read_csv(file_path, separator="\t")
    df_recovery = pl.concat([df_recovery, df_name])

In [ ]:
del df_name

In [ ]:
len(df_recovery)

## Data Cleaning

In [ ]:
# Drop columns where all values are null
cols_to_drop = (
    df_recovery.select(pl.all().is_null().all())
    .unpivot()
    .filter(pl.col("value") == True)
    .select("variable")
    .to_series()
    .to_list()
)

# Drop the identified columns from the original DataFrame
df_recovery = df_recovery.drop(cols_to_drop)

In [ ]:
# Remove all C-Returns from the data
df_recovery = df_recovery.filter(pl.col('recovery_type') != 'C-Returns')

In [ ]:
# Create target variable for propensity to waste model
df_recovery = df_recovery.with_columns(
    pl.when(pl.col('recovery_type') == 'Sales').then(pl.lit(0))
    .otherwise(pl.lit(1)).alias('recovery')
)

In [ ]:
df_recovery.filter(pl.col('country') != pl.col('marketplace_id'))

In [ ]:
# Remove marketplace_id b/c it's a duplicate of country
df_recovery = df_recovery.drop(['marketplace_id'])

In [ ]:
# Reorder the columns
df_recovery = df_recovery.select([
 'hashed_fc',
 'year',
 'month',
 'week',
 'gl_product_group',
 'product_type',
 'macro_category',
 'item_disposition_code',
 'reason_code',
 'application_name',
 'is_stranded',
 'reason_code_type',
 'reason_code_stranded',
 'stranded_potential_issue',
 'is_hazmat',
 'units',
 'cogs',
 'weight',
 'country',
 'country_state',
 'zip_code',
 'site_type',
 'site_category',
 'recovery',
 'recovery_type'])

In [ ]:
destination = "s3://msds-26.2-data/clean/combined_recovery_data.parquet"
df_recovery.write_parquet(destination)

## Aggregation and Feature Engineering

In [ ]:
df_recovery = pl.read_parquet("s3://msds-26.2-data/clean/combined_recovery_data.parquet")

In [ ]:
# Remove 3000 isntances of missing gl_product_group
df_recovery = df_recovery.filter(pl.col('gl_product_group').is_not_null())

In [ ]:
df_recovery.filter(pl.col('recovery') == 0)['reason_code'].value_counts()
# Only products that enter the recovery funnel have reason code - might not be usable for propensity to waste model

In [ ]:
df_recovery.filter(pl.col('recovery') == 0)['is_stranded'].value_counts()
# Only products that enter the recovery funnel are stranded

In [ ]:
df_recovery = df_recovery.with_columns(cogs_per_unit=pl.col('cogs')/pl.col('units'),
                                       cogs_per_weight=pl.col('cogs')/pl.col('weight'))

In [ ]:
# Aggregate to the site-week-gl level
df_recovery_agg = df_recovery.group_by(['hashed_fc', 'year', 'month', 'week', 'gl_product_group']).agg(
    pl.len().alias('num_records'),
    ((pl.col('reason_code') == 'E').sum()/pl.len()).alias('proportion_reason_code_E'),
    ((pl.col('reason_code') == 'O').sum()/pl.len()).alias('proportion_reason_code_O'),
    ((pl.col('macro_category') == 'RETAIL').sum()/pl.len()).alias('proportion_macro_category_RETAIL'),
    ((pl.col('macro_category') == 'FBA').sum()/pl.len()).alias('proportion_macro_category_FBA'),
    ((pl.col('is_hazmat') == 'O').sum()/pl.len()).alias('proportion_hazmat'),
    ((pl.col('product_type') == 'Food').sum()/pl.len()).alias('proportion_food'),
    ((pl.col('product_type') == 'Non Food').sum()/pl.len()).alias('proportion_non_food'),
    ((pl.col('product_type') == 'Pet Food').sum()/pl.len()).alias('proportion_pet_food'),
    ((pl.col('is_stranded') == 'Y').sum()/pl.len()).alias('proportion_stranded'),
    ((pl.col('reason_code_type') == 'LLSI').sum()/pl.len()).alias('proportion_reason_code_type_LLSI'),
    ((pl.col('reason_code_type') == '\\N').sum()/pl.len()).alias('proportion_reason_code_type_N'),
    ((pl.col('reason_code_type') == 'MCF').sum()/pl.len()).alias('proportion_reason_code_type_MCF'),
    ((pl.col('reason_code_type') == 'ALSI').sum()/pl.len()).alias('proportion_reason_code_type_ALSI'),
    pl.col('units').sum().alias('units_total'),
    pl.col('cogs').sum().alias('cogs_total'),
    pl.col('weight').sum().alias('weight_total'),
    pl.col('cogs_per_unit').mean().alias('cogs_per_unit_mean'),
    pl.col('cogs_per_unit').std().alias('cogs_per_unit_std'),
    pl.col('cogs_per_weight').mean().alias('cogs_per_weight_mean'),
    pl.col('cogs_per_weight').std().alias('cogs_per_weight_std'),
    pl.min('country'),
    pl.min('country_state'),
    pl.min('zip_code'),
    pl.min('site_type'),
    pl.min('site_category'),
    (pl.sum('recovery')/pl.len()).alias('proportion_recovery')
).sort(by='num_records', descending=True)

In [ ]:
group_keys = ["hashed_fc", "gl_product_group"]

value_cols = [
    "units_total",
    "cogs_total",
    "weight_total",
    "cogs_per_unit_mean",
    "cogs_per_unit_std",
    "cogs_per_weight_mean",
    "cogs_per_weight_std",
    "proportion_recovery",
]

total_cols = ["units_total", "cogs_total", "weight_total"]

rate_cols = [
    "cogs_per_unit_mean",
    "cogs_per_unit_std",
    "cogs_per_weight_mean",
    "cogs_per_weight_std",
    "proportion_recovery",
]

In [ ]:
# Create ISO week-date
df = df_recovery_agg.with_columns(
    pl.date(pl.col("year"), 1, 4)
    .dt.truncate("1w")
    .dt.offset_by(
        (pl.col("week").cast(pl.Int32) - 1).cast(pl.Utf8) + "w"
    )
    .alias("week_date")
)

In [ ]:
# Build complete week grid
full_weeks = (
    df.group_by(group_keys)
      .agg([
          pl.min("week_date").alias("start"),
          pl.max("week_date").alias("end"),
      ])
      .with_columns(
          pl.date_ranges("start", "end", interval="1w").alias("week_date")
      )
      .explode("week_date")
)

In [ ]:
# Join and Sort
df = (
    full_weeks
    .join(df, on=group_keys + ["week_date"], how="left")
    .sort(group_keys + ["week_date"])
)

In [ ]:
# Rebuild time columns
df = df.with_columns([
    pl.col("week_date").dt.year().alias("year"),
    pl.col("week_date").dt.week().alias("week"),
    pl.col("week_date").dt.month().alias("month"),
    (pl.col("week_date").dt.year() * 100 +
     pl.col("week_date").dt.week()).alias("year_week")
])

In [ ]:
# Lag and rolling features
exprs = []

for c in value_cols:
    base = pl.col(c)

    exprs.extend([
        # last week
        base.shift(1).over(group_keys).alias(f"{c}_last_week"),

        # last 4 weeks
        base.shift(1).rolling_mean(4, min_samples=2).over(group_keys).alias(f"{c}_last4wk_mean"),
        base.shift(1).rolling_std(4, min_samples=2).over(group_keys).alias(f"{c}_last4wk_std"),

        # last 12 weeks
        base.shift(1).rolling_mean(12, min_samples=6).over(group_keys).alias(f"{c}_last12wk_mean"),
        base.shift(1).rolling_std(12, min_samples=6).over(group_keys).alias(f"{c}_last12wk_std"),
    ])

df = df.with_columns(exprs)

In [ ]:
# Trend features
trend_exprs = []

for c in value_cols:
    col = pl.col(c)

    trend_exprs.extend([
        (col - col.shift(1))
            .over(group_keys)
            .alias(f"{c}_mom_1wk"),

        ((col / col.shift(1)) - 1)
            .over(group_keys)
            .alias(f"{c}_mom_1wk_pct"),

        (col - col.shift(4))
            .over(group_keys)
            .alias(f"{c}_mom_4wk"),

        (
            col.shift(1).rolling_mean(4, min_samples=2)
            - col.shift(5).rolling_mean(4, min_samples=2)
        )
        .over(group_keys)
        .alias(f"{c}_slope_4wk"),
    ])

df = df.with_columns(trend_exprs)

In [ ]:
# Get history of gl at site
df = df.with_columns(
    (
        pl.col('week').cum_count()
        .over(["hashed_fc", "gl_product_group"])
        - 1
    ).alias("weeks_since_first_appearance")
)

In [ ]:
df.filter((pl.col('hashed_fc') =='0000c60d3b14250deb1312b21a448a53591a6668f530c817b37934ace57e2574') & (pl.col('gl_product_group') == 86)).select(
    ['year', 'month', 'week', 'units_total', 'units_total_last_week',
 'units_total_last4wk_mean',
 'units_total_last4wk_std',
 'units_total_last12wk_mean',
 'units_total_last12wk_std',
    'units_total_mom_1wk',
     'units_total_mom_1wk_pct',
 'units_total_mom_4wk',
 'units_total_slope_4wk',
'weeks_since_first_appearance'])

In [ ]:
destination = "s3://msds-26.2-data/clean/combined_recovery_data_aggregated_with_new_features.parquet"
df.write_parquet(destination)